In [ ]:
image_dir = "/content/images"
image_files = os.listdir(image_dir)

all_features = []
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pipe = pipeline(task="image-feature-extraction", model_name="google/vit-base-patch16-224", device=DEVICE)

for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    image = Image.open(image_path).convert("RGB")
    output = pipe(image)
    np.array(output)
    all_features.append(output)



In [ ]:
all_features = np.array(all_features)
num_images, *num_features = all_features.shape
all_features = all_features.reshape(num_images, -1)


In [ ]:
# best_score = -1
# best_k = -1

# for k in range(2, 11):  # Try different numbers of clusters from 2 to 10
#     agg_clustering = AgglomerativeClustering(n_clusters=k)
#     cluster_labels = agg_clustering.fit_predict(all_features)
#     silhouette_avg = silhouette_score(all_features, cluster_labels)
#     print(f"For k={k}, silhouette score: {silhouette_avg}")
#     if silhouette_avg > best_score:
#         best_score = silhouette_avg
#         best_k = k

# print(f"Best number of clusters (k): {best_k}")

In [ ]:
k = 10

kmeans = KMeans(n_clusters=k)
kmeans.fit(all_features)

cluster_labels = kmeans.labels_

for i, image_file in enumerate(image_files):
    plt.title(cluster_labels[i])
    plt.imshow(plt.imread(os.path.join(image_dir, image_file)))
    plt.show()

In [ ]:
agg_clustering = AgglomerativeClustering(n_clusters=10)
cluster_labels = agg_clustering.fit_predict(all_features)

# for i, image_file in enumerate(image_files):
#     # plt.title(cluster_labels[i])
#     # plt.imshow(plt.imread(os.path.join(image_dir, image_file)))
#     # plt.show()
cluster_dict = {}
for cluster_num in range(10):
  cluster_dict[cluster_num] = [image_files[i] for i, label in enumerate(cluster_labels) if label == cluster_num]

print(cluster_dict)


In [ ]:
import numpy as np
import cv2
from sklearn.cluster import KMeans,AgglomerativeClustering
import os
import matplotlib.pyplot as plt
import torch
from transformers import pipeline
from PIL import Image
from sklearn.metrics import silhouette_score


In [ ]:
def cluster_images(image_dir, model_name="google/vit-base-patch16-224"):

    image_files = os.listdir(image_dir)

    all_features = []
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pipe = pipeline(task="image-feature-extraction", model_name=model_name, device=DEVICE)

    for image_file in image_files:
        image_path = os.path.join(image_dir, image_file)
        image = Image.open(image_path).convert("RGB")
        output = pipe(image)
        all_features.append(output)

    all_features = np.array(all_features)

    num_images, *num_features = all_features.shape
    all_features = all_features.reshape(num_images, -1)

    best_k=10
    agg_clustering = AgglomerativeClustering(n_clusters=best_k)
    cluster_labels = agg_clustering.fit_predict(all_features)

    cluster_dict = {}
    for cluster_num in range(best_k):
        cluster_dict[cluster_num] = [image_files[i] for i, label in enumerate(cluster_labels) if label == cluster_num]
    return cluster_dict

image_dir = "/content/images"
cluster_dict = cluster_images(image_dir)
for cluster_num, images in cluster_dict.items():
    print(f"Cluster {cluster_num}: {images}")


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from transformers import pipeline
from sklearn.cluster import AgglomerativeClustering

def cluster_images(image_dir, model_name="google/vit-base-patch16-224"):
    image_files = os.listdir(image_dir)

    all_features = []
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pipe = pipeline(task="image-feature-extraction", model_name=model_name, device=DEVICE)

    for image_file in image_files:
        image_path = os.path.join(image_dir, image_file)
        image = Image.open(image_path).convert("RGB")
        output = pipe(image)
        all_features.append(output)

    all_features = np.array(all_features)

    num_images, *num_features = all_features.shape
    all_features = all_features.reshape(num_images, -1)

    best_k = 10
    agg_clustering = AgglomerativeClustering(n_clusters=best_k)
    cluster_labels = agg_clustering.fit_predict(all_features)

    cluster_dict = {}
    cluster_centroids_dict = {}  # Dictionary to store centroids for each cluster
    for cluster_num in range(best_k):
        cluster_dict[cluster_num] = [image_files[i] for i, label in enumerate(cluster_labels) if label == cluster_num]
        cluster_features = all_features[cluster_labels == cluster_num]
        cluster_centroid = np.mean(cluster_features, axis=0)
        cluster_centroids_dict[cluster_num] = cluster_centroid

    return cluster_dict, cluster_centroids_dict

image_dir = "/content/images"
cluster_dict, cluster_centroids_dict = cluster_images(image_dir)




In [ ]:
def predict_cluster(new_image_path, cluster_centroids_dict, model_name="google/vit-base-patch16-224"):

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pipe = pipeline(task="image-feature-extraction", model_name=model_name, device=DEVICE)
    new_image_features = pipe(new_image_path)

    new_image_features = np.array(new_image_features)
    new_image_features = new_image_features.reshape(1, -1)

    distances = {}
    for cluster_num, centroid in cluster_centroids_dict.items():
        distance = np.linalg.norm(new_image_features - centroid)
        distances[cluster_num] = distance
        print(f"Cluster {cluster_num}: Distance = {distance}")

    predicted_cluster = min(distances, key=distances.get)

    return predicted_cluster



new_image_path = "/content/loha2.jpg"
predicted_cluster = predict_cluster(new_image_path, cluster_centroids_dict)
print("Predicted Cluster:", predicted_cluster)